# Week 9 Data source lab
Research Method of Data Science · TIC3002 · Iegor Vyshnevskyi · Fall 2026

**Question:** Can these records support a descriptive comparison of recycling and household waste across districts during 2021–2024?

The district data are **fictional teaching data**, invented for this exercise. They do not describe real places. They deliberately contain problems. They are not World Bank data.

**Learning objectives:** import without losing identifiers, inspect the records, document provenance and metadata, implement validation rules, and explain a source decision.

**Core time:** about 40 minutes in class, followed by work on your project. Optional API section: later exploration or instructor demonstration.

**Prerequisites:** a browser, Google account if using Colab, and the Week 7 concepts of unit, measurement, coverage and missingness. No previous API experience required.

**Files:** none to download for the core exercise. The notebook creates the practice CSV from text embedded below. **Software:** Python 3 and pandas. Colab normally supplies pandas. For a local Jupyter installation, install pandas in your environment if needed.

## How to open and save
1. Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and select this `.ipynb` file. Menu wording can change.
2. Save a personal copy in Drive before editing. Run code cells from top to bottom using the triangle beside each cell or Shift+Enter.
3. Edit the markdown answer cells by double-clicking them. Run the cell to display your text.
4. A variable exists only after its defining cell runs. If something is undefined, rerun the earlier setup cells.
5. Before submission, restart the session/kernel and run all core cells again. Save and download the notebook. Colab session files are temporary.

**Do not upload private survey records, identifiable participants, credentials, or restricted project data to an unapproved environment.** You can document a restricted source without loading it here.


## 1 Setup and a reproducible working folder
Run this cell. It prints the working directory so you can find the files. It does not mount Drive or access the internet. The core exercise writes only to `w9_lab/`.

`Path` manages file locations. `pandas` reads and checks tables. A SHA-256 hash is a fingerprint of file bytes; it helps detect a changed file, but does not establish accuracy or authenticity.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import pandas as pd

ROOT = Path.cwd() / "w9_lab"
RAW = ROOT / "raw"
DOCS = ROOT / "documentation"
for folder in [RAW, DOCS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", ROOT)
print("Python:", sys.version.split()[0], "pandas:", pd.__version__)


Working folder: c:\Users\vyshn\OneDrive - kdis.ac.kr\Documents\GitHub\Sogang\2026\Fall\Research Method of Data Science\W9\w9_lab
Python: 3.13.4 pandas: 2.3.0


## 2 The source note comes before the table
**Provider:** fictional District Environment Office. **Version:** teaching v1. **Expected coverage:** NA, EA and WA, each once per year from 2021 to 2024. **Unit:** district-year.

**Definitions:** `households` = number of registered households at year-end. `waste_tonnes` = household waste collected during the year in metric tonnes. `recycling_pct` = percentage of collected household waste recorded as recycled, on a 0–100 scale.

**Important limits:** registered households and collected waste may have different coverage. These totals do not directly measure individual household behaviour. The source supplies no household income or causal design.

**Codes:** `NA` is the valid district code North Area. `EA` = East Area. `WA` = West Area. `..` = suppressed value. `-999` = unavailable waste amount. Neither marker means zero.

**Rights for this invented dataset:** the instructor-created practice CSV may be reused and adapted with attribution to this course. This permission does not apply to external data or all course materials.

**Predict before running:** Could a default CSV importer mistake `NA` for missing data? What does one row represent?


In [2]:
CSV_TEXT = """district_code,year,households,waste_tonnes,recycling_pct
NA,2021,1000,220,35
NA,2022,1020,230,37
NA,2023,1040,241,..
NA,2024,1060,250,43
EA,2021,800,180,40
EA,2022,810,182,41
EA,2023,820,-999,42
EA,2024,830,190,44
WA,2021,600,150,45
WA,2022,610,151,46
WA,2024,630,154,140
EA,2022,810,182,41
"""
raw_path = RAW / "district_environment_teaching_v1.csv"
raw_path.write_bytes(CSV_TEXT.encode("utf-8"))
raw_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
source_record = {
    "title": "Fictional district environment records",
    "provider": "Fictional District Environment Office",
    "source_type": "Invented teaching data embedded in this notebook",
    "original_url": None,
    "version": "teaching_v1",
    "file_created_utc": datetime.now(timezone.utc).isoformat(),
    "observation_period": "2021-2024",
    "unit_of_observation": "district-year",
    "expected_districts": ["NA", "EA", "WA"],
    "expected_years": [2021, 2022, 2023, 2024],
    "access_route": "Embedded text written unchanged to a CSV file",
    "usage_permission": "Reuse/adaptation with attribution to TIC3002 course",
    "raw_file": str(raw_path.relative_to(ROOT)),
    "sha256": raw_hash,
}
print("Created practice file:", raw_path.name)
print("SHA-256:", raw_hash)
print("Creation date is NOT an observation date or an external retrieval date.")


Created practice file: district_environment_teaching_v1.csv
SHA-256: 9e4a0686de598891029612b19581849329de955278c6f44d616e95d8b5f7bca6
Creation date is NOT an observation date or an external retrieval date.


## 3 Import without guessing identifier or missing-value meanings
First inspect a default import. Then create the working import with **all columns as strings** and default missing-code detection disabled. This preserves identifiers and the original markers for inspection.

We do not overwrite the raw file. A later working copy will use numeric types where appropriate.


In [3]:
default_import = pd.read_csv(raw_path)
print("Default import rows with a missing district:",
      int(default_import["district_code"].isna().sum()))

raw = pd.read_csv(raw_path, dtype="string", keep_default_na=False)
print("Preserved district codes:", raw["district_code"].unique().tolist())
print("Shape (rows, columns):", raw.shape)
print(raw.head(6).to_string(index=False))
print("\nImported types:\n", raw.dtypes.to_string())


Default import rows with a missing district: 4
Preserved district codes: ['NA', 'EA', 'WA']
Shape (rows, columns): (12, 5)
district_code year households waste_tonnes recycling_pct
           NA 2021       1000          220            35
           NA 2022       1020          230            37
           NA 2023       1040          241            ..
           NA 2024       1060          250            43
           EA 2021        800          180            40
           EA 2022        810          182            41

Imported types:
 district_code    string[python]
year             string[python]
households       string[python]
waste_tonnes     string[python]
recycling_pct    string[python]


**Checkpoint 1.** You should see 12 rows and 5 columns. The preserved import contains the district code `NA`. The default import treats that code as missing in four rows.

**Your answer:** Why would losing those four identifiers matter for a district comparison? Replace this text with two sentences.


## 4 A data dictionary records meaning and rules
A data dictionary describes each field. It is separate from the source log: the log identifies the source, while the dictionary explains the columns.

Inspect the table, then add any project-specific definitions you would need. These rules follow the fictional source note; they are not universal rules for all datasets.


In [4]:
dictionary = pd.DataFrame([
    ["district_code", "District identifier", "category", "NA, EA, WA", "none expected"],
    ["year", "Observation year", "integer", "2021-2024", "none expected"],
    ["households", "Registered households at year-end", "households", "positive integer", "none expected"],
    ["waste_tonnes", "Household waste collected during year", "metric tonnes", ">=0", "-999 unavailable"],
    ["recycling_pct", "Share of collected waste recorded as recycled", "percent", "0-100", ".. suppressed"],
], columns=["variable", "definition", "unit_or_type", "valid_values", "missing_codes"])
print(dictionary.to_string(index=False))


     variable                                    definition  unit_or_type     valid_values    missing_codes
district_code                           District identifier      category       NA, EA, WA    none expected
         year                              Observation year       integer        2021-2024    none expected
   households             Registered households at year-end    households positive integer    none expected
 waste_tonnes         Household waste collected during year metric tonnes              >=0 -999 unavailable
recycling_pct Share of collected waste recorded as recycled       percent            0-100    .. suppressed


## 5 A typed working copy and transparent flags
Apply only the documented missing codes. Numeric conversion errors should become flags, not disappear.

This is a validation step. We will **not** silently remove duplicates, impute missing values, or change 140 to an assumed value. Week 10 develops cleaning decisions.


In [5]:
work = raw.copy(deep=True)
missing_codes = {"waste_tonnes": ["-999"], "recycling_pct": [".."]}
parse_flags = {}
for col in ["year", "households", "waste_tonnes", "recycling_pct"]:
    tokens = raw[col].mask(raw[col].isin(missing_codes.get(col, [])), pd.NA)
    numbers = pd.to_numeric(tokens, errors="coerce")
    parse_flags[col] = tokens.notna() & numbers.isna()
    work[col] = numbers
parse_report = pd.DataFrame(parse_flags)
print("Unexpected numeric parse failures by column:")
print(parse_report.sum().to_string())
print("\nDocumented missing values by column:")
print(work.isna().sum().to_string())


Unexpected numeric parse failures by column:
year             0
households       0
waste_tonnes     0
recycling_pct    0

Documented missing values by column:
district_code    0
year             0
households       0
waste_tonnes     1
recycling_pct    1


## 6 Validation has several layers
Run the report. Each rule has an explicit pass/fail outcome. A failed rule is a reason to investigate, not a Python error to hide.

**Predict:** Can a table have 12 rows and still fail to contain all 12 expected district-year combinations?


In [6]:
KEY = ["district_code", "year"]
expected_columns = {"district_code", "year", "households", "waste_tonnes", "recycling_pct"}
duplicate_mask = work.duplicated(KEY, keep=False)
percent_bad = work["recycling_pct"].notna() & ~work["recycling_pct"].between(0, 100)
household_bad = work["households"].notna() & (
    (work["households"] <= 0) | (work["households"] % 1 != 0))
waste_bad = work["waste_tonnes"].notna() & (work["waste_tonnes"] < 0)
year_bad = work["year"].notna() & (
    ~work["year"].between(2021, 2024) | (work["year"] % 1 != 0))
expected_keys = pd.MultiIndex.from_product(
    [["NA", "EA", "WA"], [2021, 2022, 2023, 2024]], names=KEY)
actual_keys = pd.MultiIndex.from_frame(work[KEY].drop_duplicates())
absent_keys = expected_keys.difference(actual_keys)
unexpected_keys = actual_keys.difference(expected_keys)
checks = [
    ("Expected columns present", set(work.columns) == expected_columns),
    ("Keys have no missing values", not work[KEY].isna().any().any()),
    ("District codes match source note", work["district_code"].isin(["NA", "EA", "WA"]).all()),
    ("Years are valid integers", not year_bad.any() and work["year"].notna().all()),
    ("Numeric tokens parse as documented", not parse_report.any().any()),
    ("District-year key is unique", not duplicate_mask.any()),
    ("All expected district-years present", len(absent_keys) == 0),
    ("No unexpected district-years", len(unexpected_keys) == 0),
    ("Households are positive integers", not household_bad.any() and work["households"].notna().all()),
    ("Observed waste is nonnegative", not waste_bad.any()),
    ("Observed recycling share is 0-100", not percent_bad.any()),
    ("Waste and recycling are complete", not work[["waste_tonnes", "recycling_pct"]].isna().any().any()),
]
validation = pd.DataFrame(checks, columns=["rule", "passes"])
print(validation.to_string(index=False))
print("\nRows involved in duplicate keys:", int(duplicate_mask.sum()))
print("Extra repeated records:", int(work.duplicated(KEY).sum()))
print("Missing expected district-years:", list(absent_keys))
print("Unexpected district-years:", list(unexpected_keys))


                               rule  passes
           Expected columns present    True
        Keys have no missing values    True
   District codes match source note    True
           Years are valid integers    True
 Numeric tokens parse as documented    True
        District-year key is unique   False
All expected district-years present   False
       No unexpected district-years    True
   Households are positive integers    True
      Observed waste is nonnegative    True
  Observed recycling share is 0-100   False
   Waste and recycling are complete   False

Rows involved in duplicate keys: 2
Extra repeated records: 1
Missing expected district-years: [('WA', 2023)]
Unexpected district-years: []


## 7 Inspection makes the flags interpretable
The following cells show suspect records and summarize coverage. Counts of nonmissing values can include suspect and repeated records, so they do not establish analysis readiness.


In [7]:
print("Rows with duplicate keys:")
print(work.loc[duplicate_mask].to_string(index=False))
print("\nRows with recycling shares outside the stated range:")
print(work.loc[percent_bad].to_string(index=False))
print("\nRows with documented missing measurements:")
print(work.loc[work[["waste_tonnes", "recycling_pct"]].isna().any(axis=1)].to_string(index=False))
print("\nPer-district coverage:")
coverage = work.groupby("district_code", dropna=False).agg(
    records=("year", "size"), unique_years=("year", "nunique"),
    first_year=("year", "min"), last_year=("year", "max"))
print(coverage.to_string())


Rows with duplicate keys:
district_code  year  households  waste_tonnes  recycling_pct
           EA  2022         810           182             41
           EA  2022         810           182             41

Rows with recycling shares outside the stated range:
district_code  year  households  waste_tonnes  recycling_pct
           WA  2024         630           154            140

Rows with documented missing measurements:
district_code  year  households  waste_tonnes  recycling_pct
           NA  2023        1040           241           <NA>
           EA  2023         820          <NA>             42

Per-district coverage:
               records  unique_years  first_year  last_year
district_code                                              
EA                   5             4        2021       2024
NA                   4             4        2021       2024
WA                   3             3        2021       2024


**Checkpoint 2.** Explain the findings rather than just repeating a count.

1. What is the duplicated key? Why should you investigate before deleting it?
2. Which expected district-year is absent? How is this different from a missing value within an existing row?
3. Which value violates its documented scale? Why would automatically dividing it by 100 be unsafe?
4. Which missing value is suppressed and which is unavailable? Why does the distinction matter?

**Your answers:** Replace this text. There is no need to repair every issue in this lesson.


## 8 A source decision and issue log
A useful issue log says what failed, how it could affect the research, and what you will do next. Keeping a suspect record pending review can be more defensible than guessing a correction.

The example below documents one issue. Add the other issues before submission.


In [8]:
issue_log = pd.DataFrame([
    {
        "issue": "Duplicate EA-2022 key",
        "evidence": "Two identical records in the raw file",
        "research_consequence": "Could give EA-2022 extra weight in summaries",
        "next_action": "Check source export/version; keep raw file unchanged",
        "status": "unresolved",
    },
    # Add rows for absent coverage, documented missing values, and the out-of-range share.
])
print(issue_log.to_string(index=False))

source_record["readiness"] = "Not ready for unrestricted comparison; source issues remain"
source_record["known_limits"] = [
    "Fictional teaching data, no real-world inference",
    "Duplicate key, absent district-year, documented missing values and out-of-range value",
    "Registered households and collected waste may differ in coverage",
    "Aggregate district records do not identify household behaviour or causal effects",
]


                issue                              evidence                         research_consequence                                          next_action     status
Duplicate EA-2022 key Two identical records in the raw file Could give EA-2022 extra weight in summaries Check source export/version; keep raw file unchanged unresolved


**Your source decision:** In three to five sentences, state what this dataset could answer if the issues were resolved, what remains unsupported, and whether you would accept it, accept with conditions, or reject it for the stated question. Explain the conditions.

**Your verification note:** If you used AI for code or wording, identify the help and checks you performed. Otherwise write “No AI assistance used.” AI output does not establish a source’s provenance or license.


## 9 Save the documentation and verify raw-file integrity
This exports documentation, not a cleaned or analysis-ready dataset. Timestamped source metadata can change between runs. The raw CSV hash should stay the same.

A small ZIP is convenient for downloading from Colab. It contains only the invented practice file and documentation created here.


In [9]:
import zipfile

assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == raw_hash, "Raw file changed"
(DOCS / "source_log.json").write_text(json.dumps(source_record, indent=2), encoding="utf-8")
dictionary.to_csv(DOCS / "data_dictionary.csv", index=False)
validation.to_csv(DOCS / "validation_report.csv", index=False)
issue_log.to_csv(DOCS / "issue_log.csv", index=False)
(DOCS / "environment.json").write_text(
    json.dumps({"python": sys.version.split()[0], "pandas": pd.__version__}, indent=2), encoding="utf-8")
practice_bundle = ROOT / "practice_audit.zip"
with zipfile.ZipFile(practice_bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for file in sorted([raw_path] + list(DOCS.glob("*"))):
        z.write(file, arcname=file.relative_to(ROOT))
print("Raw-file integrity check passed.")
print("Practice bundle:", practice_bundle)
print("In Colab use the Files sidebar, open w9_lab, and download practice_audit.zip.")
print("Also save/download this notebook, including your written answers.")


Raw-file integrity check passed.
Practice bundle: c:\Users\vyshn\OneDrive - kdis.ac.kr\Documents\GitHub\Sogang\2026\Fall\Research Method of Data Science\W9\w9_lab\practice_audit.zip
In Colab use the Files sidebar, open w9_lab, and download practice_audit.zip.
Also save/download this notebook, including your written answers.


## 10 Apply the workflow to your individual project
**In class: about 20 minutes plus peer feedback.** Start with a non-sensitive CSV export and its documentation. If you use Excel, export a CSV copy and record that conversion; preserve the original workbook. CSV encoding and decimal conventions may need explicit settings. Do not use `on_bad_lines='skip'` merely to conceal a malformed file.

If your data are restricted, use the practice table for code and write the project audit from approved metadata. If your final study is qualitative, audit a relevant quantitative contextual source without changing the approved project method.

**Your project source log**
- Research question and intended use of this dataset
- Original producer and distributor/portal, dataset/table/indicator ID and URL
- Observation period, release/update/version where known, actual retrieval time
- Unit, population/geography, frequency, definitions, units and missing codes
- Exact access route/query/filters, license/permission URL and checked date
- What you may share, where the raw file is stored, and major coverage limits

**Your project rules**
Define the expected columns, unique key, valid types/ranges, and expected coverage **from documentation**. Then inspect at least three applicable checks. State unresolved issues honestly.

**Your first action:** Fill in the configuration below. `None` keeps the optional import disabled so Run All still succeeds. The output stays separate from the practice dataset.


In [ ]:
PROJECT_CSV = None  # Example after upload: Path("my_public_data.csv")
PROJECT_KEY = []     # Example: ["country_code", "year"]
PROJECT_NA_CODES = []  # Fill from source documentation; preserve originals first.

if PROJECT_CSV is None:
    print("Project import disabled. Use practice data or set an approved local CSV path.")
else:
    project_path = Path(PROJECT_CSV)
    if not project_path.is_file():
        print("File not found:", project_path, "Check the Files sidebar/current folder.")
    else:
        project_raw = pd.read_csv(project_path, dtype="string", keep_default_na=False)
        print("Project shape:", project_raw.shape)
        print(project_raw.head().to_string(index=False))
        print("Columns:", project_raw.columns.tolist())
        if PROJECT_KEY and set(PROJECT_KEY).issubset(project_raw.columns):
            print("Rows with repeated project keys:", int(project_raw.duplicated(PROJECT_KEY, keep=False).sum()))
        print("Next: document types, missing codes, ranges, coverage and permissions.")
        print("No automatic cleaning or validation approval has been applied.")


**Project response template**

Question and source fit: …

Unit and expected key: …

Provenance and version: …

Usage/sharing conditions and evidence: …

Check 1 / result / implication: …

Check 2 / result / implication: …

Check 3 / result / implication: …

Readiness decision and next action: …

Peer feedback and one revision: …

AI-use disclosure and verification: …


## 11 Optional live World Bank API example
**Not required to finish Thursday’s lab.** This part illustrates a real external source and needs internet access. It requests GDP per capita in constant 2015 US dollars (`NY.GDP.PCAP.KD`) for Korea and Vietnam, 2019–2023, from World Development Indicators (`source=2`). It cannot answer the fictional district question or establish individual wellbeing.

The [indicator page](https://data.worldbank.org/indicator/NY.GDP.PCAP.KD) identifies the measure and displays CC BY 4.0. Read the [dataset terms and third-party exceptions](https://www.worldbank.org/ext/en/legal/terms-conditions/datasets) for your actual source. Attribution should identify the World Bank, dataset/indicator and underlying providers as relevant.

**API** means application programming interface: software requests a structured response. **JSON** is a text format containing labelled values. A provider may split records into pages. A successful HTTP response alone does not prove the data are complete or valid.

Set `RUN_LIVE_API = True` only when you want to run the example. If a request fails, the notebook reports the failure and does not substitute fictional data. Numerical results may change when the provider revises its series.


In [11]:
from urllib.request import urlopen, Request
from urllib.parse import urlencode

RUN_LIVE_API = True  # Set False to disable live World Bank API access in this teaching notebook.

# Fetching and pagination are separated so the parser can be tested without a network.
def fetch_json(url):
    request = Request(url, headers={"User-Agent": "TIC3002 teaching example"})
    with urlopen(request, timeout=20) as response:
        body = response.read()
    return body, json.loads(body.decode("utf-8"))

def collect_wb_pages(base_url, params, fetcher=fetch_json, max_pages=5):
    records, archive = [], []
    page = 1
    while True:
        url = base_url + "?" + urlencode({**params, "page": page})
        body, payload = fetcher(url)
        if not (isinstance(payload, list) and len(payload) == 2
                and isinstance(payload[0], dict) and isinstance(payload[1], list)):
            raise ValueError("Unexpected API schema or provider error. Inspect response.")
        meta, rows = payload
        pages = int(meta["pages"])
        if int(meta["page"]) != page or pages < 1 or pages > max_pages:
            raise ValueError("Unexpected page metadata or teaching request exceeds page limit.")
        archive.append({"url": url, "body": body, "metadata": meta})
        records.extend(rows)
        if page >= pages:
            if len(records) != int(meta["total"]):
                raise ValueError("Record count differs from advertised total.")
            break
        page += 1
    return records, archive

live_data = None
if not RUN_LIVE_API:
    print("Live API disabled. Core lab requires no internet.")
else:
    base_url = "https://api.worldbank.org/v2/country/KOR;VNM/indicator/NY.GDP.PCAP.KD"
    params = {"date": "2019:2023", "format": "json", "per_page": 50, "source": 2}
    try:
        records, archive = collect_wb_pages(base_url, params)
        api_dir = ROOT / "world_bank_optional"
        api_dir.mkdir(exist_ok=True)
        for i, item in enumerate(archive, start=1):
            (api_dir / f"response_page_{i}.json").write_bytes(item["body"])
        live_data = pd.DataFrame([
            {"country_code": r["countryiso3code"], "year": int(r["date"]),
             "indicator_code": r["indicator"]["id"], "gdp_per_capita_constant_2015_usd": r["value"]}
            for r in records])
        live_data["gdp_per_capita_constant_2015_usd"] = pd.to_numeric(
            live_data["gdp_per_capita_constant_2015_usd"], errors="raise")
        live_data = live_data.sort_values(["country_code", "year"]).reset_index(drop=True)
        keys = ["country_code", "year", "indicator_code"]
        expected_live_keys = pd.MultiIndex.from_product(
            [["KOR", "VNM"], range(2019, 2024), ["NY.GDP.PCAP.KD"]], names=keys)
        actual_live_keys = pd.MultiIndex.from_frame(live_data[keys].drop_duplicates())
        print(live_data.to_string(index=False))
        print("Repeated keys:", int(live_data.duplicated(keys).sum()))
        print("Missing expected keys:", list(expected_live_keys.difference(actual_live_keys)))
        print("Unexpected keys:", list(actual_live_keys.difference(expected_live_keys)))
        print("Missing values:", int(live_data["gdp_per_capita_constant_2015_usd"].isna().sum()))
        print("Unexpected nonpositive observed values:", int((live_data["gdp_per_capita_constant_2015_usd"].dropna() <= 0).sum()))
        retrieval = {
            "provider": "World Bank", "dataset": "World Development Indicators",
            "indicator": "NY.GDP.PCAP.KD", "indicator_label": "GDP per capita (constant 2015 US$)",
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "queries": [a["url"] for a in archive],
            "provider_response_metadata": [a["metadata"] for a in archive],
            "raw_response_hashes": [hashlib.sha256(a["body"]).hexdigest() for a in archive],
            "indicator_page": "https://data.worldbank.org/indicator/NY.GDP.PCAP.KD",
            "terms_page": "https://www.worldbank.org/ext/en/legal/terms-conditions/datasets",
            "license_note": "Indicator page displays CC BY 4.0; check current source-specific terms before sharing",
        }
        (api_dir / "retrieval_log.json").write_text(json.dumps(retrieval, indent=2), encoding="utf-8")
        live_data.to_csv(api_dir / "gdp_per_capita_working.csv", index=False)
        print("Saved raw responses, working table and retrieval log to", api_dir)
    except Exception as exc:
        live_data = None
        print("Live API example could not complete:", type(exc).__name__, str(exc))
        print("No substitute data used. Continue with the core teaching dataset.")


country_code  year indicator_code  gdp_per_capita_constant_2015_usd
         KOR  2019 NY.GDP.PCAP.KD                      33496.311382
         KOR  2020 NY.GDP.PCAP.KD                      33215.929893
         KOR  2021 NY.GDP.PCAP.KD                      34792.939721
         KOR  2022 NY.GDP.PCAP.KD                      35809.013811
         KOR  2023 NY.GDP.PCAP.KD                      36347.703639
         VNM  2019 NY.GDP.PCAP.KD                       3241.076490
         VNM  2020 NY.GDP.PCAP.KD                       3303.169498
         VNM  2021 NY.GDP.PCAP.KD                       3358.217348
         VNM  2022 NY.GDP.PCAP.KD                       3617.663113
         VNM  2023 NY.GDP.PCAP.KD                       3772.360291
Repeated keys: 0
Missing expected keys: []
Unexpected keys: []
Missing values: 0
Unexpected nonpositive observed values: 0
Saved raw responses, working table and retrieval log to c:\Users\vyshn\OneDrive - kdis.ac.kr\Documents\GitHub\Sogang\2026\Fall\Re

**Optional interpretation:** Why is GDP per capita in constant prices different from a current-dollar measure? Does this series measure household income, inequality, or happiness? What additional documentation would you consult before a comparison?

**Your answer:** …


## 12 Final reproducibility and submission check
- Fill in the practice answers and issue log. Complete the project audit or document a specific access obstacle and next action.
- Keep `RUN_LIVE_API=False` for the core reproducibility check. Set the optional project path only if the file is available in the restarted session.
- Restart the kernel/session, then run all core cells in order. The validation report should still show the intentional source issues. They are not a failed notebook execution.
- Save your notebook and the relevant documentation. Download session files you need to keep.
- Submit through the posted Cyber Campus instructions. No new grade weight or deadline is set here.

## Sources and attribution
Teaching data and exercises: created for TIC3002, Research Method of Data Science. Practice-data permission appears in Section 2. The API example uses original provider documentation, not invented data.
- [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api)
- [World Bank API call structures](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)
- [World Bank indicator](https://data.worldbank.org/indicator/NY.GDP.PCAP.KD)
- [World Bank dataset terms](https://www.worldbank.org/ext/en/legal/terms-conditions/datasets)
- [pandas read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

Software and documentation reviewed October 2026. Check current source definitions and terms when you use them.
